In [23]:
from langchain_huggingface import ChatHuggingFace , HuggingFaceEndpoint
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema.runnable import RunnableSequence
from langchain.tools import BaseTool 
from typing import Type
from pydantic import BaseModel , Field
from langchain.memory import ChatMessageHistory 
from langchain_core.runnables.history import RunnableWithMessageHistory


load_dotenv()

llm = HuggingFaceEndpoint(
    repo_id = "google/gemma-2-2b-it",
    # repo_id="deepseek-ai/DeepSeek-R1-0528",

    # huggingfacehub_api_token="YOUR_HF_TOKEN",
    huggingfacehub_api_token="YOUR_HF_TOKEN",
    task = "text-generation"
)

model = ChatHuggingFace(llm = llm)


In [24]:
prompt = PromptTemplate(
    input_variables=['history' , 'question'],
    template="""
You are a helpful assistant. Use the following conversation history to answer the user's question.

Conversation History:
{history}

User : {question}


"""

)

In [25]:
parser = StrOutputParser()
chain = prompt | model | parser

In [26]:
store = {}
# this will store the session of each user , it is not persistant if u close the program and run it again all the sessions will be removed so history wont be visible, for persistent we can use Databases

In [27]:
def get_session_history(session_id : str):
    if session_id not in store :
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# we’re storing a ChatMessageHistory instance (object) as the value for each session_id key

# store = {
#     "user123": <ChatMessageHistory instance at 0x...>,
#     "user456": <ChatMessageHistory instance at 0x...>
# }

In [28]:
runnable  = RunnableWithMessageHistory(
    chain,
    get_session_history = get_session_history,
    input_messages_key='question', # the key that contains user input
    history_messages_key= 'history' # where chat history goes in the prompt
)

# onyl question & history are tracked as pairs in the MEMORY , if we have multiple inputs from the user say
# input_variables = ["topic" , "food" , "question"]

# if we do input_messages_key = 'question'
# only "question" input is tracked in the memory 
# downside is we can only track the "question" input in runnablewithmessagehistory.
# if we need to track say "food" tooo along with "question" , we need to add topic food with question in the prompt
# EXAMPLE :-  combined_input = f"Food topic: {food}\nQuestion: {question}"

In [29]:
response1 = runnable.invoke(
    {"question" : "hi what is your name , i have a dog named bunny , he like to eat cat food though"},
    config={'configurable' : {"session_id" : "user1"}}
)

In [30]:
print(response1)

Hi there!  My name is Gemma.  That's interesting - I bet Bunny is curious! 😊  Do you have any other pets? 



In [31]:
response1 = runnable.invoke(
    {"question" : "What was my dog's name again?"},
    config={'configurable' : {"session_id" : "user1"}}
)
# see that we are using the same session id , same session id loads the data for the same session id , if we close the program and rerun it the store will be empty , therefore we need to use persistent memory like databases

In [32]:
print(response1)

Your dog's name is Bunny. 😊🐕 



In [ ]:
# 2️⃣ Why you don’t pass history manually when invoking
# When you call:

# runnable.invoke({"topic": "sports", "mood": "excited", "question": "Who won the World Cup?"}, config={"configurable": {"session_id": "user1"}})
# LangChain does the following internally:

# Calls get_session_history("user1") → returns the ChatMessageHistory object for this session.

# Adds the current question ("Who won the World Cup?") as a HumanMessage to that history.

# Converts the entire chat history (all previous Human & AI messages) into a string to fill the {history} placeholder in your prompt.

# Passes your chain a dictionary like:

# {
#     "topic": "sports",
#     "mood": "excited",
#     "question": "Who won the World Cup?",
#     "history": "<formatted string of past messages>"  # filled automatically
# }
# ✅ So you don’t need to manually send history — RunnableWithMessageHistory automatically injects it from the session memory.

# 3️⃣ Conceptual Flow
# User input ("question") -> RunnableWithMessageHistory
#             |
#             v
#     Add question to ChatMessageHistory
#             |
#             v
#     Retrieve all previous messages -> format into string
#             |
#             v
#         Inject as 'history' in chain input
#             |
#             v
#          Call your chain
#             |
#             v
#    LLM outputs -> stored as AIMessage in history

In [33]:
# Conversation Flow with RunnableWithMessageHistory
# User sends a question

# "Who won the World Cup?"
# → This is passed to RunnableWithMessageHistory via input_messages_key="question".

# Fetch past chat history

# LangChain calls your get_session_history(session_id) function.

# It retrieves all previous Human + AI messages for that session.

# Inject history into the prompt

# All previous messages are formatted as a string.

# This string is placed in the {history} placeholder of your prompt template.

# Send to the LLM

# The chain now receives a full input like:

# topic: sports
# mood: excited
# history: 
# Human: Tell me about last year's World Cup
# AI: Country X won the 2024 World Cup
# Human: Who won this year?
# question: Who won the World Cup?
# LLM generates an answer

# Example output: "This year, Country Y won the World Cup."

# Store LLM output in memory

# The output is automatically added to the same session’s ChatMessageHistory as an AIMessage.

# Next question

# Steps 1–6 repeat, so the conversation “remembers” all previous turns.